In [1]:
import argparse
import json
import os
import torch
import traceback
import numpy as np
from datetime import datetime
from typing import Dict, List, Any
from pathlib import Path
from tqdm import tqdm
from argparse import Namespace

# Set correct directory pathing
import os
import sys

# Import project modules
sys.path.insert(0, '/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA')
from rdma.rdrag.entity import LLMRDExtractor, RetrievalEnhancedRDExtractor,MultiIterativeRDExtractor,IterativeLLMRDExtractor
from rdma.utils.embedding import EmbeddingsManager
from rdma.hporag.context import ContextExtractor
from rdma.utils.llm_client import LocalLLMClient, APILLMClient
from rdma.utils.setup import setup_device
from dotenv import load_dotenv

load_dotenv()

/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


True

In [2]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [3]:
SAMPLE_SIZE = 5

### GettingStarted

In [4]:
from pyhealth.datasets.mimic4 import MIMIC4NoteDataset

In [5]:
NOTE_ROOT = '/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp'

dataset = MIMIC4NoteDataset(root=NOTE_ROOT, tables=["discharge"])
note_df = dataset.global_event_df.collect().to_pandas()

Using default note config: /Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/configs/mimic4_note.yaml
Memory usage Before initializing mimic4_note: 564.0 MB
Initializing mimic4_note dataset from /Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp (dev mode: False)
Memory usage After initializing mimic4_note: 564.2 MB
No cache_dir provided. Using default cache dir: /Users/williampang/Library/Caches/pyhealth/d41554f3-03fc-5ed2-a6e5-99b796e1ae37


/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/mimic4.py:103: UserWarning: Events from discharge table only have date timestamp (no specific time). This may affect temporal ordering of events.
  warnings.warn(


In [6]:
patient_notes = (
    note_df
    .sort_values("timestamp")
    .groupby("patient_id")
    .apply(
        lambda x: dict(zip(x["timestamp"], x["discharge/text"]))
     )
    )

samples = patient_notes.sample(n=SAMPLE_SIZE, random_state=42)

/var/folders/zl/lm3qrxjd2jl17byd443y505h0000gn/T/ipykernel_7165/4148831802.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [7]:
final_notes = {
    str(patient_id): {
        str(charttime): {"note_content": text} 
        for charttime, text in notes.items()
    }
    for patient_id, notes in samples.items()
}

## Extract Rare Disease

In [8]:
args = Namespace(
      llm_type="api",
      api_config="api_config.json"
  )

def initialize_llm_client(args: argparse.Namespace):
    """Initialize appropriate LLM client based on arguments."""
    if args.llm_type == "api":
        if args.api_config:
            return APILLMClient.from_config(args.api_config)
        else:
            return APILLMClient.initialize_from_input()
    else:  # local
        return LocalLLMClient(
            model_type=args.model_type,
            device=device,
            cache_dir=args.cache_dir,
            temperature=args.temperature
        )

llm_client = initialize_llm_client(args)        

entity_extractor = LLMRDExtractor(
      llm_client=llm_client,
      system_message="You are a medical expert specializing in rare diseases."
  )

context_extractor = ContextExtractor()

In [10]:
for i, (patient_id, patient_data) in enumerate(tqdm(list(final_notes.items()), desc="Processing cases")):
    for charttime, note in patient_data.items():
        note['llm_extracted_entities'] = entity_extractor.extract_entities(note['note_content'])
        note['entity_context'] = context_extractor.extract_contexts(note['llm_extracted_entities'], note['note_content'], window_size=0)

Processing cases:   0%|                                   | 0/5 [00:00<?, ?it/s]

TOTAL_TOKENS_USED before query: 0


Processing cases:  20%|█████▍                     | 1/5 [00:01<00:04,  1.22s/it]

TOTAL_TOKENS_USED before query: 3953


Processing cases:  40%|██████████▊                | 2/5 [00:02<00:02,  1.02it/s]

...

Processing cases:  60%|████████████████▏          | 3/5 [00:02<00:01,  1.31it/s]

TOTAL_TOKENS_USED before query: 12071


Processing cases:  80%|█████████████████████▌     | 4/5 [00:10<00:03,  3.41s/it]

TOTAL_TOKENS_USED before query: 16099
TOTAL_TOKENS_USED before query: 18769


Processing cases: 100%|███████████████████████████| 5/5 [00:25<00:00,  5.06s/it]


In [15]:
final_notes['13106750']['2117-04-16 00:00:00']['entity_context']

[{'entity': 'Squamos cell carcinoma',
  'context': 'Squamos cell carcinoma of epiglottis, treated with'},
 {'entity': 'Adenocarcinoma',
  'context': 'completed ___ ___, adenocarcinoma of stage IV left lung'},
 {'entity': 'COPD', 'context': 'COPD'},
 {'entity': 'HIT', 'context': 'HIT with positive antibody assay.'},
 {'entity': 'Hypertension', 'context': 'Hypertension'},
 {'entity': 'Hyperlipidemia', 'context': 'Hyperlipidemia'},
 {'entity': 'CKD', 'context': 'CKD stage III (baseline 1.'},
 {'entity': 'GERD', 'context': 'GERD'},
 {'entity': 'Uric acid nephrolithiasis',
  'context': 'History of microscopic hematuria with known uric acid'},
 {'entity': 'Thrombocytopenia',
  'context': 'Course complicated by thrombocytopenia and was diagnosed with'},
 {'entity': 'Hematuria',
  'context': 'History of microscopic hematuria with known uric acid'},
 {'entity': 'Pneumonia',
  'context': 'found to have sepesis secondary to bilateral pneumonia and'},
 {'entity': 'Sepsis', 'context': 'sepsis.'},
 